# Lab 3 — zadania 2 i 3

Dane `people` są w `../data` (pliki `00.json`–`99.json` z przykładu w labie). Daty ważności kart mają postać `miesiąc/rok`. Karta wygasa z końcem tego miesiąca.

In [2]:
import json
import os
from datetime import date
from glob import glob

import dask.bag as db
import dask.dataframe as dd

DATAPATH = os.path.join("..", "data")
TODAY = date.today()

# Konkretna lista plików. Wzorzec z "?" na Windows nie rozwija się w ścieżce Daska.
source_files = sorted(
    path
    for path in glob(os.path.join(DATAPATH, "*.json"))
    if not os.path.basename(path).startswith("expired_")
)
people = db.read_text(source_files).map(json.loads)
len(source_files), people

(100, dask.bag<loads, npartitions=100>)

**Zadanie 2.** Z wszystkich plików zostają rekordy z przeterminowaną kartą. `repartition(10)` ustawia 10 partycji, a `to_textfiles` zapisuje je jako `expired_*.json`. W tym zbiorze daty kończą się na 2025, więc 22 października 2026 każda karta jest już po terminie.

In [3]:
def card_expired(record, today=TODAY):
    month_text, year_text = record["credit-card"]["expiration-date"].split("/")
    year = 2000 + int(year_text)
    month = int(month_text)
    return (year, month) < (today.year, today.month)

expired = people.filter(card_expired).repartition(10)

for path in glob(os.path.join(DATAPATH, "expired_*.json")):
    os.remove(path)

expired.map(json.dumps).to_textfiles(os.path.join(DATAPATH, "expired_*.json"))

expired_files = sorted(glob(os.path.join(DATAPATH, "expired_*.json")))
len(expired_files), expired.count().compute()

(10, 1000000)

**Zadanie 3.** `adults` to osoby z `age >= 18`. Zagnieżdżone `name`, `address` i `credit-card` są spłaszczone do zwykłych kolumn, potem Bag przechodzi w Dask DataFrame i w jeden plik Parquet.

In [3]:
def is_adult(record):
    return record["age"] >= 18

def flatten_person(record):
    first_name, last_name = record["name"]
    address = record["address"]
    card = record["credit-card"]
    return {
        "age": record["age"],
        "first_name": first_name,
        "last_name": last_name,
        "occupation": record["occupation"],
        "telephone": record["telephone"],
        "address": address["address"],
        "city": address["city"],
        "credit_card_number": card["number"],
        "credit_card_expiration": card["expiration-date"],
    }

adults = people.filter(is_adult)
adults_df = adults.map(flatten_person).to_dataframe().repartition(npartitions=1)

parquet_path = os.path.join(DATAPATH, "adults.parquet")
adults_df.to_parquet(parquet_path, overwrite=True)

saved = dd.read_parquet(parquet_path)
saved.npartitions, len(saved.columns), saved.head(3)

(1,
 9,
    age first_name last_name          occupation        telephone  \
 0   34    Armanda     Mills           Carpenter  +1-989-305-6247   
 1   75    Ignacia     Weber  Purchasing Manager     +13473563718   
 2   79     Blythe      Cook   Hearing Therapist  +1-216-175-3338   
 
               address      city   credit_card_number credit_card_expiration  
 0  545 Balceta Center      Troy  2522 9133 6408 4771                  07/16  
 1   466 Holyoke Plaza  Gastonia  5576 9865 5929 4551                  03/25  
 2   100 Waithman Lake    Mobile  2247 8120 8555 5206                  06/22  )